# Day 3 · Morning exercises (Exercise A)

**09:45–11:10 · then everyone back in the room at 11:10 for Peer review A**

Bayesian statistics for PhD researchers, Aalborg University 2026. Work in your group and keep your own copy of this notebook. Select **R (bayes)** and run the cells in order from a fresh kernel. The code uses R with the tidyverse (dplyr for summaries, ggplot2 for figures) and the `posterior` package for the effective sample size. There is no Stan this morning. The afternoon continues in `day-3-exercises-afternoon.ipynb`.

## Plan for this session

| Part | What | Guide time |
|---|---|---|
| A1 | Monte Carlo with independent draws | 15 min |
| A2 | Run and read a Metropolis sampler | 20 min |
| A3 | A chain that hardly moves | 15 min |
| A4 | **Challenge, no template:** your own sampler for a rate | 25 min |
| — | Exit response and your group's question set | 10 min |

**Roles this block:** coder, checker, two question writers, interviewer, reporter. Rotate them in the afternoon.

**The example for the whole day.** We ran n = 10 trials and saw y = 7 successes; p is the probability of success in one trial. With the flat prior Beta(1, 1) the exact posterior is **Beta(8, 4)** (Day 2, Exercise A1). Because we know the exact answer, we can check every simulation against it.

In [ ]:
options(tidyverse.quiet = TRUE)
library(tidyverse)   # dplyr for summaries, ggplot2 for figures
set.seed(2026)
options(width = 95, repr.plot.width = 8, repr.plot.height = 5, repr.plot.bg = "white")

## A1 (15 min): Monte Carlo with independent draws

**The idea** (*slides "Integrals become averages", "The estimate itself is random" and "Where does √S come from?"*). Draw S values from the posterior and use averages of the draws:

- posterior mean ≈ `mean(draws)`;
- P(p > 0.5 | y) ≈ `mean(draws > 0.5)`, the fraction of draws above 0.5.

These estimates are random: another set of draws gives slightly different numbers. The **Monte Carlo standard error** (MCSE) says how much they vary, and it can be computed from one set of draws:

MCSE = sd(h) / √S, where h holds the values we average (the draws themselves for the mean; the TRUE/FALSE values `draws > 0.5`, counted as 1/0, for the probability).

**Tasks.**

1. Draw S = 5,000 values from Beta(8, 4). Estimate the posterior mean and P(p > 0.5 | y), and compute the MCSE of each.
2. Compare with the exact values `8 / 12` and `1 - pbeta(0.5, 8, 4)`. Are the differences of the size of the MCSE?
3. Run the cell again with `set.seed(1)`, `set.seed(2)` and `set.seed(3)` at the top. How much do the estimates change?
4. The posterior SD of p is about 0.13; the MCSE of the mean is about 0.002. What does each number describe? Which one would you report as the uncertainty about p? (*slide "Two kinds of uncertainty"*)

In [ ]:
S <- 5000
draws <- rbeta(S, 8, 4)
h <- draws > 0.5             # TRUE/FALSE, counted as 1/0

# Task 1: estimates and their MCSEs
# mean(draws);  sd(draws) / sqrt(S)
# mean(h);      sd(h) / sqrt(S)

# Task 2: exact values
# 8 / 12;  1 - pbeta(0.5, 8, 4)

**Your answer:** write a few sentences here.

## A2 (20 min): run and read a Metropolis sampler

**The idea** (*slides "The Metropolis algorithm", "One step, drawn" and "The Metropolis sampler in R"*). Pretend we cannot use `rbeta()`. We can still compute likelihood × prior at any p. The Metropolis sampler walks through values of p: propose a nearby value, move there with probability min(1, ratio of heights), otherwise stay, and record the current value at every iteration.

The first cell defines the sampler. It is the same as on the slide, now for our data (7 of 10, flat prior).

**Tasks.**

1. Read the code of `metropolis()`. Which line makes the proposal? Which line computes the log of the height ratio? Which line accepts the proposal? Where is the current value recorded after a *rejection*?
2. Run 6,000 iterations with `step = 0.3`, starting at 0.5. Drop the first 1,000 iterations as warmup (*slide "Where to start, and when to start counting"*).
3. Make a trace plot of the kept values, and a histogram with the exact posterior density on top. The exact posterior comes from the Day 2 beta–binomial update: prior Beta(a, b) and y successes in n trials give Beta(a + y, b + n − y), here Beta(1 + 7, 1 + 10 − 7) = **Beta(8, 4)**. In R its density is `dbeta(p, shape1 = 8, shape2 = 4)`.
4. Compare the mean and P(p > 0.5) of the kept values with the exact values from A1.
5. Why must the chain record the current value again after a rejection? What would go wrong if we kept only the accepted proposals?

In [ ]:
# log(likelihood x prior) for 7 successes in 10 trials, prior Beta(1, 1)
log_post <- function(p) {
  if (p <= 0 || p >= 1) return(-Inf)          # outside (0, 1): height 0
  dbinom(7, 10, p, log = TRUE) + dbeta(p, 1, 1, log = TRUE)
}

metropolis <- function(n_iter, step, start) {
  chain <- numeric(n_iter)
  current <- start
  for (i in 1:n_iter) {
    proposal  <- current + rnorm(1, mean = 0, sd = step)
    log_ratio <- log_post(proposal) - log_post(current)
    if (log(runif(1)) < log_ratio) current <- proposal
    chain[i] <- current
  }
  chain
}

# fraction of iterations where the chain moved
acceptance_rate <- function(chain) mean(diff(chain) != 0)

In [ ]:
set.seed(2026)
chain <- metropolis(6000, step = 0.3, start = 0.5)
acceptance_rate(chain)
kept <- chain[1001:6000]     # drop the first 1,000 as warmup

# Task 3: trace plot and histogram
# tibble(iteration = 1001:6000, p = kept) |>
#   ggplot(aes(x = iteration, y = p)) + geom_line()
# tibble(p = kept) |>
#   ggplot(aes(x = p)) +
#   geom_histogram(aes(y = after_stat(density)), binwidth = 0.02) +
#   stat_function(fun = dbeta, args = list(shape1 = 8, shape2 = 4), colour = "red")   # exact Beta(1 + 7, 1 + 3)

# Task 4: mean(kept), mean(kept > 0.5)

**Your answer:** write a few sentences here.

## A3 (15 min): a chain that hardly moves

**The idea** (*slides "Effective sample size (ESS)", "Autocorrelation and ESS of our chains" and "The step size matters"*). Values of a Markov chain are dependent: neighbouring values are similar. The **effective sample size** (ESS) is the number of independent draws that would give the same precision. For a chain, the MCSE uses the ESS instead of S:

MCSE = sd(kept) / √ESS.   In R: `posterior::ess_basic(kept)`.

**Tasks.**

1. Run a second chain with `step = 0.005`, starting at 0.1, for the same 6,000 iterations, and drop the first 1,000.
2. Plot the traces of both chains in one figure (one panel per chain).
3. Make a small table for both chains: acceptance rate, mean, ESS and MCSE. Compare the means with the exact 0.667.
4. The tiny-step chain accepts almost every proposal. Why is it still bad?
5. Is its MCSE a trustworthy description of its error? Compare MCSE and actual error, and look at its trace.

*Optional:* try `step = 3`. What do the trace, acceptance rate and ESS show now?

In [ ]:
set.seed(2026)
tiny <- metropolis(6000, step = 0.005, start = 0.1)
kept_tiny <- tiny[1001:6000]

# Task 2: both traces, one panel each
# traces <- bind_rows(
#   tibble(iteration = 1001:6000, p = kept,      sampler = "step 0.3"),
#   tibble(iteration = 1001:6000, p = kept_tiny, sampler = "step 0.005"))
# ggplot(traces, aes(x = iteration, y = p)) + geom_line() + facet_wrap(~ sampler, ncol = 1)

# Task 3: one row per chain
# ess_good <- posterior::ess_basic(kept)
# ess_tiny <- posterior::ess_basic(kept_tiny)
# tibble(sampler    = c("step 0.3", "step 0.005"),
#        acceptance = c(acceptance_rate(chain), acceptance_rate(tiny)),
#        mean       = c(mean(kept), mean(kept_tiny)),
#        ess        = c(ess_good, ess_tiny),
#        mcse       = c(sd(kept) / sqrt(ess_good), sd(kept_tiny) / sqrt(ess_tiny)))

**Your answer:** write a few sentences here.

## A4 (25 min): challenge, no template — your own sampler for a rate

*Slides "The Metropolis algorithm", "The Metropolis sampler in R", "Effective sample size (ESS)", "The step size matters" and "Where to start, and when to start counting".* A new posterior, and no starter code: plan the steps in your group, then write them.

**The problem.** You counted the emails arriving in your personal inbox in eight one-hour periods: **3, 5, 2, 4, 6, 3, 4, 5** (made-up numbers). Model: the counts are Poisson with rate λ (emails per hour), and the prior is λ ~ Gamma(2, 1). From Day 2, the exact posterior is Gamma(2 + 32, 1 + 8) = **Gamma(34, 9)**, so you can check your sampler against it.

**Tasks.**

1. Write a function for the log of likelihood × prior of λ, and a Metropolis sampler that uses it. λ must stay positive.
2. Run **four chains** of 5,000 iterations from the starting values 0.5, 2, 6 and 10, and drop the first 1,000 iterations of each. Try step sizes 0.05, 1 and 5. For each, report the acceptance rate, R-hat and the ESS of the four chains together. Plot the traces of the four chains for the step size you choose, one colour per chain.
3. With your chosen step size, estimate the posterior mean of λ and P(λ > 4 | y), each with its Monte Carlo standard error. Are they within 2 MCSE of the exact values `34 / 9` and `1 - pgamma(4, 34, 9)`?

<details><summary><b>Hints</b> (open only when you are stuck)</summary>

- **Log posterior:** `sum(dpois(emails, lambda, log = TRUE)) + dgamma(lambda, 2, 1, log = TRUE)`, and `-Inf` when `lambda <= 0` (as `log_post()` above returns `-Inf` outside (0, 1)).
- **Sampler:** A2's `metropolis()` shows the loop; yours calls your own log posterior.
- **Four chains:** a matrix with one column per chain, `chains <- matrix(NA, 4000, 4)`, filled in a `for` loop over the four starting values.
- **R-hat and ESS of several chains:** `posterior::rhat(chains)` and `posterior::ess_basic(chains)` accept such a matrix (iterations × chains).
- **Trace plot:** `tibble(iteration = rep(1:4000, 4), chain = factor(rep(1:4, each = 4000)), lambda = as.vector(chains))`, then `geom_line()` with `colour = chain`.
- **MCSE of a probability:** P(λ > 4) is the mean of the indicator `kept > 4`, so its MCSE is `sd(indicator) / sqrt(ESS of the indicator)`.

</details>

In [ ]:
# A4: your code

**Your answer:** write a few sentences here.

## Exit response

Write 80–120 words: The exact posterior is known, yet a simulation gives a slightly different answer. Where does the difference come from, and why is a high acceptance rate not enough to trust a chain?

**Your exit response:**

## Your group's question set

Question writers: write **three** questions about this block, each with your own worked solution:

1. a **concept** question (what does a result mean?),
2. a **computation** question (a number someone can check in R),
3. a **pitfall** question (a mistake that is easy to make).

The checker reviews the solutions. The reporter notes what your group found hardest, to ask in the plenary.

**Concept question:**

**Our solution:**

**Computation question:**

**Our solution:**

**Pitfall question:**

**Our solution:**

**Hardest open question (reporter):**

---
**Everyone back in the room at 11:10 for Peer review A.** Save this notebook (Ctrl/Cmd + S). After lunch, open `day-3-exercises-afternoon.ipynb`.